# File Manager Notebook

| Cell | Operation |
|------|-----------|
| **1** | Install & Imports |
| **2** | All helper functions (run once) |
| **3** | ▶ Extract / Unzip any archive |
| **4** | ▶ Move files or folders |
| **5** | ▶ Copy files or folders |
| **6** | ▶ Rename a file or folder |
| **7** | ▶ Delete files or folders |
| **A** | Appendix — Browse local directory & disk usage |

## 1 — Install & Imports

In [1]:
%pip install lz4 py7zr zstandard tqdm --quiet

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import shutil
import tarfile
import zipfile
import gzip
import bz2
import lzma
import time
import threading
from pathlib import Path
from datetime import datetime

from tqdm import tqdm

print("✓ Core imports ready")

# ── Optional archive backends ──────────────────────────────────────────────
_lz4_ok = _zstd_ok = _7z_ok = False
try:
    import lz4.frame as lz4frame
    _lz4_ok = True;  print("✓ lz4  (handles .lz4, .tar.lz4)")
except ImportError:
    print("✗ lz4  — run: pip install lz4")

try:
    import zstandard as zstd
    _zstd_ok = True; print("✓ zstandard  (handles .zst, .tar.zst)")
except ImportError:
    print("✗ zstandard — run: pip install zstandard")

try:
    import py7zr
    _7z_ok = True;   print("✓ py7zr  (handles .7z)")
except ImportError:
    print("✗ py7zr — run: pip install py7zr")

✓ Core imports ready
✓ lz4  (handles .lz4, .tar.lz4)
✓ zstandard  (handles .zst, .tar.zst)
✓ py7zr  (handles .7z)


## 2 — Helper Functions

Run this cell once. All operation cells below depend on it.

In [3]:
# ══════════════════════════════════════════════════════════════════════════════
# HELPERS
# ══════════════════════════════════════════════════════════════════════════════

def _human_size(n: int) -> str:
    for u in ("B", "KB", "MB", "GB", "TB"):
        if n < 1024: return f"{n:.1f} {u}"
        n /= 1024
    return f"{n:.1f} PB"


def _box(lines: list[str], width: int = 70) -> str:
    sep = "─" * (width - 2)
    def row(s): return f"║  {s:<{width-4}}║"
    out = [f"╔{sep}╗"]
    for l in lines:
        out.append(f"╠{sep}╣" if l == "---" else row(l))
    out.append(f"╚{sep}╝")
    return "\n".join(out)


def _confirm(prompt: str) -> bool:
    """Ask the user to type 'yes' to confirm a destructive operation."""
    ans = input(f"  {prompt} [yes / no]: ").strip().lower()
    return ans == "yes"


def _archive_type(path: Path) -> str:
    """
    Detect archive type from the file name.
    Returns one of: tar.lz4 | tar.zst | tar.gz | tar.bz2 | tar.xz | tar
                    zip | gz | bz2 | xz | lz4 | zst | 7z | rar | unknown
    """
    name = path.name.lower()
    for ext in (".tar.lz4", ".tar.zst", ".tar.gz", ".tar.bz2",
                ".tar.xz", ".tar.lzma", ".tgz", ".tbz2",
                ".zip", ".7z", ".rar",
                ".gz", ".bz2", ".xz", ".lzma", ".lz4", ".zst", ".tar"):
        if name.endswith(ext):
            return ext.lstrip(".")
    return "unknown"


# ── Extract ────────────────────────────────────────────────────────────────

def _extract_tar_lz4(src: Path, dest: Path):
    """Stream-decompress LZ4 → pipe into tarfile → extract with progress."""
    if not _lz4_ok:
        raise RuntimeError("lz4 not installed. Run: pip install lz4")

    file_size = src.stat().st_size
    dest.mkdir(parents=True, exist_ok=True)

    print(f"Decompressing (LZ4) and extracting …")
    with (
        open(src, "rb") as raw,
        lz4frame.open(raw, mode="rb") as lz4_stream,
        tqdm(total=file_size, unit="B", unit_scale=True,
             unit_divisor=1024, desc="Reading") as bar,
    ):
        class _TrackedStream:
            """Wraps the lz4 stream to update the progress bar on reads."""
            def read(self, n=-1):
                chunk = lz4_stream.read(n)
                bar.update(raw.tell() - bar.n)
                return chunk
            def readinto(self, b):
                n = lz4_stream.readinto(b)
                bar.update(raw.tell() - bar.n)
                return n
            # tarfile needs these attributes
            readable  = lambda s: True
            writable  = lambda s: False
            seekable  = lambda s: False
            tell      = lambda s: lz4_stream.tell()

        with tarfile.open(fileobj=_TrackedStream(), mode="r|") as tf:
            tf.extractall(path=dest)


def _extract_tar_zst(src: Path, dest: Path):
    """Stream-decompress Zstandard → tarfile."""
    if not _zstd_ok:
        raise RuntimeError("zstandard not installed. Run: pip install zstandard")
    file_size = src.stat().st_size
    dest.mkdir(parents=True, exist_ok=True)
    dctx = zstd.ZstdDecompressor()
    with open(src, "rb") as fh:
        with tqdm(total=file_size, unit="B", unit_scale=True,
                  unit_divisor=1024, desc="Reading") as bar:
            stream = dctx.stream_reader(fh)
            with tarfile.open(fileobj=stream, mode="r|") as tf:
                for member in tf:
                    tf.extract(member, path=dest)
                    bar.update(fh.tell() - bar.n)


def _extract_zip(src: Path, dest: Path):
    dest.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(src) as zf:
        members = zf.infolist()
        total   = sum(m.file_size for m in members)
        with tqdm(total=total, unit="B", unit_scale=True,
                  unit_divisor=1024, desc="Extracting") as bar:
            for member in members:
                zf.extract(member, path=dest)
                bar.update(member.file_size)


def _extract_7z(src: Path, dest: Path):
    if not _7z_ok:
        raise RuntimeError("py7zr not installed. Run: pip install py7zr")
    dest.mkdir(parents=True, exist_ok=True)
    with py7zr.SevenZipFile(src, mode="r") as archive:
        file_list = archive.getnames()
        print(f"  Extracting {len(file_list)} items …")
        archive.extractall(path=dest)


def _extract_single(src: Path, dest_file: Path, open_fn):
    """Decompress a single-file archive (.gz, .bz2, .xz, .lz4, .zst)."""
    file_size = src.stat().st_size
    dest_file.parent.mkdir(parents=True, exist_ok=True)
    with open_fn(src, "rb") as f_in, open(dest_file, "wb") as f_out:
        with tqdm(total=file_size, unit="B", unit_scale=True,
                  unit_divisor=1024, desc="Decompressing") as bar:
            while chunk := f_in.read(1 << 20):  # 1 MB chunks
                f_out.write(chunk)
                bar.update(len(chunk))


def extract(
    source_path: str | Path,
    extract_to:  str | Path,
    overwrite:   bool = False,
) -> Path:
    """
    Extract / decompress any archive into *extract_to*.

    Supported formats
    -----------------
    .tar.lz4  .tar.zst  .tar.gz  .tar.bz2  .tar.xz  .tar
    .zip  .7z  .gz  .bz2  .xz  .lz4  .zst

    Parameters
    ----------
    source_path : Path to the archive file.
    extract_to  : Directory to extract into.
    overwrite   : If True, remove existing destination before extracting.
    """
    src  = Path(source_path)
    dest = Path(extract_to)

    if not src.exists():
        raise FileNotFoundError(f"Archive not found: {src}")

    atype = _archive_type(src)
    if atype == "unknown":
        raise ValueError(f"Unsupported archive type: {src.name}")

    if dest.exists() and not overwrite:
        print(_box([
            "EXTRACT SKIPPED", "---",
            f"Archive  : {src.name}",
            f"Dest     : {dest}",
            "---",
            "⏭  Destination already exists — set OVERWRITE = True to re-extract",
        ]))
        return dest

    if dest.exists() and overwrite:
        shutil.rmtree(dest) if dest.is_dir() else dest.unlink()

    t0 = time.time()
    print(f"Archive  : {src.name}  ({_human_size(src.stat().st_size)})")
    print(f"Format   : {atype}")
    print(f"Dest     : {dest}\n")

    if atype == "tar.lz4":
        _extract_tar_lz4(src, dest)
    elif atype == "tar.zst":
        _extract_tar_zst(src, dest)
    elif atype in ("tar.gz", "tgz", "tar.bz2", "tbz2", "tar.xz",
                   "tar.lzma", "tar"):
        dest.mkdir(parents=True, exist_ok=True)
        with tarfile.open(src) as tf:
            members = tf.getmembers()
            with tqdm(total=len(members), unit="file", desc="Extracting") as bar:
                for m in members:
                    tf.extract(m, path=dest)
                    bar.update(1)
    elif atype == "zip":
        _extract_zip(src, dest)
    elif atype == "7z":
        _extract_7z(src, dest)
    elif atype == "gz":
        _extract_single(src, dest / src.stem, gzip.open)
    elif atype == "bz2":
        _extract_single(src, dest / src.stem, bz2.open)
    elif atype in ("xz", "lzma"):
        _extract_single(src, dest / src.stem, lzma.open)
    elif atype == "lz4":
        if not _lz4_ok:
            raise RuntimeError("pip install lz4")
        _extract_single(src, dest / src.stem, lz4frame.open)
    elif atype == "zst":
        if not _zstd_ok:
            raise RuntimeError("pip install zstandard")
        dctx = zstd.ZstdDecompressor()
        _extract_single(src, dest / src.stem,
                        lambda p, m: dctx.stream_reader(open(p, "rb")))

    elapsed  = time.time() - t0
    out_size = sum(f.stat().st_size for f in dest.rglob("*") if f.is_file())
    n_files  = sum(1 for f in dest.rglob("*") if f.is_file())
    n_dirs   = sum(1 for f in dest.rglob("*") if f.is_dir())

    print(_box([
        "EXTRACTION COMPLETE", "---",
        f"Archive      : {src.name}",
        f"Format       : {atype}",
        f"Extracted to : {dest}",
        "---",
        f"Files        : {n_files}",
        f"Sub-folders  : {n_dirs}",
        f"Total size   : {_human_size(out_size)}",
        f"Time taken   : {elapsed:.1f}s",
    ]))
    return dest


# ── Move ───────────────────────────────────────────────────────────────────

def move(
    source_path: str | Path,
    dest_path:   str | Path,
    overwrite:   bool = False,
) -> Path:
    """
    Move a file or folder to *dest_path*.

    If *dest_path* is an existing directory the item is moved inside it.
    Set overwrite=True to replace an existing destination file.
    """
    src  = Path(source_path)
    dest = Path(dest_path)
    if not src.exists():
        raise FileNotFoundError(f"Not found: {src}")

    final = dest / src.name if dest.is_dir() else dest
    if final.exists() and not overwrite:
        raise FileExistsError(
            f"Destination exists: {final}\nSet overwrite=True to replace it.")
    if final.exists():
        shutil.rmtree(final) if final.is_dir() else final.unlink()

    final.parent.mkdir(parents=True, exist_ok=True)
    shutil.move(str(src), str(final))

    kind = "folder" if final.is_dir() else "file"
    print(_box([
        "MOVE COMPLETE", "---",
        f"Type   : {kind}",
        f"From   : {src}",
        f"To     : {final}",
    ]))
    return final


# ── Copy ───────────────────────────────────────────────────────────────────

def copy(
    source_path: str | Path,
    dest_path:   str | Path,
    overwrite:   bool = False,
) -> Path:
    """
    Copy a file or entire folder tree to *dest_path*.
    If *dest_path* is an existing directory the item is copied inside it.
    """
    src  = Path(source_path)
    dest = Path(dest_path)
    if not src.exists():
        raise FileNotFoundError(f"Not found: {src}")

    final = dest / src.name if dest.is_dir() else dest
    if final.exists() and not overwrite:
        raise FileExistsError(
            f"Destination exists: {final}\nSet overwrite=True to replace it.")
    if final.exists():
        shutil.rmtree(final) if final.is_dir() else final.unlink()

    t0 = time.time()
    if src.is_dir():
        all_files = [f for f in src.rglob("*") if f.is_file()]
        total     = sum(f.stat().st_size for f in all_files)
        with tqdm(total=total, unit="B", unit_scale=True,
                  unit_divisor=1024, desc="Copying") as bar:
            def _copy_fn(s, d):
                r = shutil.copy2(s, d)
                bar.update(Path(s).stat().st_size)
                return r
            shutil.copytree(src, final, copy_function=_copy_fn)
        kind = "folder"
        size = _human_size(sum(f.stat().st_size for f in final.rglob("*") if f.is_file()))
    else:
        final.parent.mkdir(parents=True, exist_ok=True)
        size = _human_size(src.stat().st_size)
        with tqdm(total=src.stat().st_size, unit="B", unit_scale=True,
                  unit_divisor=1024, desc="Copying") as bar:
            with open(src, "rb") as f_in, open(final, "wb") as f_out:
                while chunk := f_in.read(1 << 20):
                    f_out.write(chunk)
                    bar.update(len(chunk))
        kind = "file"

    elapsed = time.time() - t0
    print(_box([
        "COPY COMPLETE", "---",
        f"Type   : {kind}",
        f"From   : {src}",
        f"To     : {final}",
        f"Size   : {size}",
        f"Time   : {elapsed:.1f}s",
    ]))
    return final


# ── Rename ─────────────────────────────────────────────────────────────────

def rename(
    source_path: str | Path,
    new_name:    str,
) -> Path:
    """
    Rename a file or folder in-place (same parent directory).

    Parameters
    ----------
    source_path : Current path of the file or folder.
    new_name    : New name only (not a full path), e.g. "dataset_v2".
    """
    src   = Path(source_path)
    final = src.parent / new_name
    if not src.exists():
        raise FileNotFoundError(f"Not found: {src}")
    if final.exists():
        raise FileExistsError(f"Already exists: {final}")
    src.rename(final)
    print(_box([
        "RENAME COMPLETE", "---",
        f"From : {src.name}",
        f"To   : {final.name}",
        f"Dir  : {src.parent}",
    ]))
    return final


# ── Delete ─────────────────────────────────────────────────────────────────

def delete(
    target_path: str | Path,
    confirm:     bool = True,
) -> bool:
    """
    Delete a file or folder permanently.

    Parameters
    ----------
    target_path : Path to delete.
    confirm     : If True (default), asks for interactive confirmation.
                  Set to False only when you are certain — no undo!
    """
    target = Path(target_path)
    if not target.exists():
        print(f"  ⚠  Not found (nothing deleted): {target}")
        return False

    kind = "folder" if target.is_dir() else "file"
    if target.is_dir():
        n     = sum(1 for _ in target.rglob("*") if _.is_file())
        size  = _human_size(sum(f.stat().st_size for f in target.rglob("*") if f.is_file()))
        label = f"{n} files — {size}"
    else:
        label = _human_size(target.stat().st_size)

    print(f"  ⚠  About to permanently delete {kind}: {target}")
    print(f"     Contains: {label}")

    if confirm and not _confirm("Type 'yes' to confirm deletion"):
        print("  ✗  Cancelled.")
        return False

    shutil.rmtree(target) if target.is_dir() else target.unlink()
    print(_box([
        "DELETE COMPLETE", "---",
        f"Deleted ({kind}) : {target}",
        f"Content          : {label}",
    ]))
    return True


# ── Browse / disk usage ────────────────────────────────────────────────────

def browse(
    path:      str | Path = ".",
    max_depth: int = 2,
    show_hidden: bool = False,
) -> None:
    """
    Print a tree view of *path* up to *max_depth* levels deep.
    Shows file sizes and last-modified timestamps.
    """
    root = Path(path)
    if not root.exists():
        print(f"Path not found: {root}")
        return

    def _tree(p: Path, prefix: str, depth: int):
        if depth > max_depth:
            return
        items = sorted(p.iterdir(), key=lambda x: (x.is_file(), x.name))
        if not show_hidden:
            items = [i for i in items if not i.name.startswith(".")]
        for idx, item in enumerate(items):
            is_last  = idx == len(items) - 1
            branch   = "└── " if is_last else "├── "
            ext_pre  = "    " if is_last else "│   "
            if item.is_dir():
                n_ch = sum(1 for _ in item.iterdir())
                print(f"{prefix}{branch}📁 {item.name}/  [{n_ch} items]")
                _tree(item, prefix + ext_pre, depth + 1)
            else:
                mtime = datetime.fromtimestamp(item.stat().st_mtime).strftime("%Y-%m-%d %H:%M")
                size  = _human_size(item.stat().st_size)
                print(f"{prefix}{branch}📄 {item.name}  {size:>10}  {mtime}")

    total_files = sum(1 for f in root.rglob("*") if f.is_file())
    total_size  = sum(f.stat().st_size for f in root.rglob("*") if f.is_file())
    print(f"📂 {root}")
    print(f"   {total_files} files — {_human_size(total_size)} total\n")
    _tree(root, "", 1)


def disk_usage(path: str | Path = ".") -> None:
    """Print disk usage for every immediate child of *path*, sorted by size."""
    root = Path(path)
    if not root.exists():
        print(f"Path not found: {root}")
        return

    rows = []
    for item in sorted(root.iterdir()):
        if item.is_file():
            rows.append((item.stat().st_size, "file",   item.name))
        elif item.is_dir():
            sz = sum(f.stat().st_size for f in item.rglob("*") if f.is_file())
            rows.append((sz, "folder", item.name))

    rows.sort(reverse=True)
    total = sum(r[0] for r in rows)
    print(f"\n{'Name':<45} {'Type':<8} {'Size':>12}  {'%':>6}")
    print("─" * 78)
    for sz, kind, name in rows:
        pct = sz / total * 100 if total else 0
        icon = "📁" if kind == "folder" else "📄"
        print(f"{icon} {name:<43} {kind:<8} {_human_size(sz):>12}  {pct:5.1f}%")
    print("─" * 78)
    print(f"{'Total':<45} {'':8} {_human_size(total):>12}")


print("✓ All helpers loaded — ready to use")

✓ All helpers loaded — ready to use


## 3 — Extract / Unzip

Handles: `.tar.lz4` `.tar.zst` `.tar.gz` `.tar.bz2` `.tar.xz` `.tar` `.zip` `.7z` `.gz` `.bz2` `.xz` `.lz4` `.zst`

In [14]:
# ── INPUT ──────────────────────────────────────────────────────────────────
SOURCE_ARCHIVE = "/home/taiaburrahman/dataset_manager_pro/data/wasabi/gen_ai_detector_dataset_scaled_224/human.tar.lz4"
EXTRACT_TO     = "/home/taiaburrahman/dataset_manager_pro/data/gen_ai_detector_dataset_scaled_224/human/"
OVERWRITE      = False   # True = remove existing dest and re-extract
# ───────────────────────────────────────────────────────────────────────────

extract(SOURCE_ARCHIVE, EXTRACT_TO, overwrite=OVERWRITE)

Archive  : human.tar.lz4  (17.8 GB)
Format   : tar.lz4
Dest     : /home/taiaburrahman/dataset_manager_pro/data/gen_ai_detector_dataset_scaled_224/human

Decompressing (LZ4) and extracting …


Reading: 100%|██████████| 17.8G/17.8G [04:01<00:00, 79.2MB/s]


╔────────────────────────────────────────────────────────────────────╗
║  EXTRACTION COMPLETE                                               ║
╠────────────────────────────────────────────────────────────────────╣
║  Archive      : human.tar.lz4                                      ║
║  Format       : tar.lz4                                            ║
║  Extracted to : /home/taiaburrahman/dataset_manager_pro/data/gen_ai_detector_dataset_scaled_224/human║
╠────────────────────────────────────────────────────────────────────╣
║  Files        : 336016                                             ║
║  Sub-folders  : 1                                                  ║
║  Total size   : 17.9 GB                                            ║
║  Time taken   : 242.0s                                             ║
╚────────────────────────────────────────────────────────────────────╝


PosixPath('/home/taiaburrahman/dataset_manager_pro/data/gen_ai_detector_dataset_scaled_224/human')

## 4 — Move

Move a file **or** an entire folder. If `DEST_PATH` is an existing directory, the item is moved *inside* it.

In [ ]:
# ── INPUT ──────────────────────────────────────────────────────────────────
SOURCE_PATH = "/home/taiaburrahman/dataset_manager_pro/data/extracted/genAI_2/"
DEST_PATH   = "/home/taiaburrahman/dataset_manager_pro/data/ready/"
OVERWRITE   = False
# ───────────────────────────────────────────────────────────────────────────

move(SOURCE_PATH, DEST_PATH, overwrite=OVERWRITE)

## 5 — Copy

Copy a file **or** an entire folder tree with a byte-level progress bar.

In [ ]:
# ── INPUT ──────────────────────────────────────────────────────────────────
SOURCE_PATH = "/home/taiaburrahman/dataset_manager_pro/data/ready/genAI_2/"
DEST_PATH   = "/home/taiaburrahman/dataset_manager_pro/data/backup/"
OVERWRITE   = False
# ───────────────────────────────────────────────────────────────────────────

copy(SOURCE_PATH, DEST_PATH, overwrite=OVERWRITE)

## 6 — Rename

Rename a file or folder in-place. `NEW_NAME` is just the new name, not a full path.

In [ ]:
# ── INPUT ──────────────────────────────────────────────────────────────────
SOURCE_PATH = "/home/taiaburrahman/dataset_manager_pro/data/ready/genAI_2/"
NEW_NAME    = "genAI_2_extracted"   # new name only (not a full path)
# ───────────────────────────────────────────────────────────────────────────

rename(SOURCE_PATH, NEW_NAME)

## 7 — Delete

⚠ **Permanent — no recycle bin.** You will be asked to type `yes` to confirm.

In [ ]:
# ── INPUT ──────────────────────────────────────────────────────────────────
TARGET_PATH = "/home/taiaburrahman/dataset_manager_pro/data/backup/genAI_2/"
# ───────────────────────────────────────────────────────────────────────────

delete(TARGET_PATH, confirm=True)   # confirm=False skips the prompt

## Appendix — Browse & Disk Usage

**`browse(path)`** — tree view with sizes and timestamps.  
**`disk_usage(path)`** — size ranking of every item in a directory.

In [ ]:
# ── INPUT ──────────────────────────────────────────────────────────────────
BROWSE_PATH = "/home/taiaburrahman/dataset_manager_pro/data/"
MAX_DEPTH   = 3      # how many folder levels to show
SHOW_HIDDEN = False  # include dot-files / dot-folders
# ───────────────────────────────────────────────────────────────────────────

browse(BROWSE_PATH, max_depth=MAX_DEPTH, show_hidden=SHOW_HIDDEN)

In [ ]:
# ── INPUT ──────────────────────────────────────────────────────────────────
USAGE_PATH = "/home/taiaburrahman/dataset_manager_pro/data/"
# ───────────────────────────────────────────────────────────────────────────

disk_usage(USAGE_PATH)